# CartPole Balancing ke liye RL Training

Yeh notebook [AI for Beginners Curriculum](http://aka.ms/ai-beginners) ka hissa hai. Ise [official PyTorch tutorial](https://pytorch.org/tutorials/intermediate/reinforcement_q_learning.html) aur [is Cartpole PyTorch implementation](https://github.com/yc930401/Actor-Critic-pytorch) se inspire kiya gaya hai.

Is example mein, hum RL use karke ek model ko train karenge taaki woh ek pole ko balance kar sake, jo ek cart par rakha hai aur horizontal scale par left aur right move kar sakta hai. Hum is pole ko simulate karne ke liye [OpenAI Gym](https://www.gymlibrary.ml/) environment use karenge.

> **Note**: Aap is lesson ka code locally (jaise Visual Studio Code se) chala sakte hain, is case mein simulation ek nayi window mein khulega. Jab aap code online chala rahe ho, toh aapko code mein kuch changes karne pad sakte hain, jaisa [yahan](https://towardsdatascience.com/rendering-openai-gym-envs-on-binder-and-google-colab-536f99391cc7) describe kiya gaya hai.

Hum shuruaat yeh make sure karke karenge ki Gym installed hai:

In [ ]:
import sys
!{sys.executable} -m pip install gym

Ab chaliye CartPole environment banate hain aur dekhte hain ki ise kaise operate kiya jaata hai. Ek environment ki yeh properties hoti hain:

* **Action space** woh set hai jisme saare possible actions shaamil hain jo hum simulation ke har step par le sakte hain.
* **Observation space** woh space hai jisme saare observations shaamil hain jo hum kar sakte hain.

In [ ]:
import gym

env = gym.make("CartPole-v1")

print(f"Action space: {env.action_space}")
print(f"Observation space: {env.observation_space}")

Chaliye dekhte hain simulation kaise kaam karta hai. Neeche diya gaya loop simulation ko tab tak chalata hai jab tak `env.step` termination flag `done` return nahi karta. Hum `env.action_space.sample()` use karke actions randomly choose karenge, iska matlab hai ki experiment shayad bahut jaldi fail ho jayega (CartPole environment tab khatam hota hai jab CartPole ki speed, uski position ya angle kuch limits ke bahar hote hain).

> Simulation ek nayi window mein khulega. Aap code ko kai baar chala sakte hain aur dekh sakte hain ki woh kaise behave karta hai.

In [ ]:
env.reset()

done = False
total_reward = 0
while not done:
   env.render()
   obs, rew, done, info = env.step(env.action_space.sample())
   total_reward += rew
   print(f"{obs} -> {rew}")
print(f"Total reward: {total_reward}")

Aap dekh sakte hain ki observation mein 4 numbers hote hain. Yeh hain:
- Cart ki position
- Cart ki velocity
- Pole ka angle
- Pole ki rotation rate

`rew` woh reward hai jo hamein har step par milta hai. Aap dekh sakte hain ki CartPole environment mein aapko har simulation step ke liye 1 point ka reward milta hai, aur goal total reward ko maximize karna hai, yaani CartPole ko gire bina balance mein rakhne ka time.

Reinforcement Learning ke dauran, hamara goal ek **policy** $\pi$ ko train karna hai, jo har state $s$ ke liye hamein batayegi ki kaun sa action $a$ lena hai, yaani basically $a = \pi(s)$.

Agar aap probabilistic solution chahte hain, toh aap policy ko aisa maan sakte hain jo har action ke liye probabilities ka set return karti hai, yaani $\pi(a|s)$ ka matlab hoga ki state $s$ par hamein action $a$ lene ki kitni probability hai.

## Policy Gradient Method

Sabse simple RL algorithm, jise **Policy Gradient** kehte hain, mein hum ek neural network ko train karenge taaki woh agla action predict kar sake.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

num_inputs = 4
num_actions = 2

model = torch.nn.Sequential(
    torch.nn.Linear(num_inputs, 128, bias=False, dtype=torch.float32),
    torch.nn.ReLU(),
    torch.nn.Linear(128, num_actions, bias = False, dtype=torch.float32),
    torch.nn.Softmax(dim=1)
)

Hum network ko kai experiments chalakar train karenge, aur har run ke baad apna network update karenge. Chaliye ek function define karte hain jo experiment chalayega aur results return karega (jise **trace** kehte hain) - saari states, actions (aur unki recommended probabilities), aur rewards:

In [ ]:
def run_episode(max_steps_per_episode = 10000,render=False):    
    states, actions, probs, rewards = [],[],[],[]
    state = env.reset()
    for _ in range(max_steps_per_episode):
        if render:
            env.render()
        action_probs = model(torch.from_numpy(np.expand_dims(state,0)))[0]
        action = np.random.choice(num_actions, p=np.squeeze(action_probs.detach().numpy()))
        nstate, reward, done, info = env.step(action)
        if done:
            break
        states.append(state)
        actions.append(action)
        probs.append(action_probs.detach().numpy())
        rewards.append(reward)
        state = nstate
    return np.vstack(states), np.vstack(actions), np.vstack(probs), np.vstack(rewards)

Aap bina trained network ke ek episode chala sakte hain aur dekh sakte hain ki total reward (jise episode length bhi kehte hain) bahut kam hai:

In [ ]:
s, a, p, r = run_episode()
print(f"Total reward: {np.sum(r)}")

Policy gradient algorithm ke tricky aspects mein se ek hai **discounted rewards** ka use. Idea yeh hai ki hum game ke har step par total rewards ka vector calculate karte hain, aur is process ke dauran hum shuruaati rewards ko kisi coefficient $gamma$ se discount karte hain. Hum resulting vector ko normalize bhi karte hain, kyunki hum ise apni training ko affect karne ke liye weight ke roop mein use karenge:

In [ ]:
eps = 0.0001

def discounted_rewards(rewards,gamma=0.99,normalize=True):
    ret = []
    s = 0
    for r in rewards[::-1]:
        s = r + gamma * s
        ret.insert(0, s)
    if normalize:
        ret = (ret-np.mean(ret))/(np.std(ret)+eps)
    return ret

Ab hum asli training shuru karenge! Hum 300 episodes chalayenge, aur har episode mein yeh karenge:

1. Experiment chalayein aur trace collect karein.
1. Liye gaye actions aur predicted probabilities ke beech ka difference (`gradients`) calculate karein. Difference jitna kam hoga, utna hi zyada hum sure honge ki humne sahi action liya hai.
1. Discounted rewards calculate karein aur unhe `gradients` ke saath multiply karein - isse ensure hoga ki zyada reward wale steps final result par kam reward wale steps ke mukable zyada impact daalein.
1. Hamare neural network ke liye expected target actions partly run ke dauran predicted probabilities se aur partly calculate kiye gaye `gradients` se liye jayenge. Hum `alpha` parameter use karenge yeh decide karne ke liye ki `gradients` aur rewards ko kitna dhyan mein rakha jaaye - ise *reinforcement algorithm ka learning rate* kehte hain.
1. Aakhir mein, hum apne network ko states aur expected actions par train karenge, aur is process ko repeat karenge.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

def train_on_batch(x, y):
    x = torch.from_numpy(x)
    y = torch.from_numpy(y)
    optimizer.zero_grad()
    predictions = model(x)
    loss = -torch.mean(torch.log(predictions) * y)
    loss.backward()
    optimizer.step()
    return loss

In [ ]:
alpha = 1e-4

history = []
for epoch in range(300):
    states, actions, probs, rewards = run_episode()
    one_hot_actions = np.eye(2)[actions.T][0]
    gradients = one_hot_actions-probs
    dr = discounted_rewards(rewards)
    gradients *= dr
    target = alpha*np.vstack([gradients])+probs
    train_on_batch(states,target)
    history.append(np.sum(rewards))
    if epoch%100==0:
        print(f"{epoch} -> {np.sum(rewards)}")

plt.plot(history)

Ab chaliye episode ko rendering ke saath chalate hain taaki results dekh sakein:

In [ ]:
_ = run_episode(render=True)

Umeed hai ki ab aap dekh sakte hain ki pole ab kaafi achhe se balance kar leta hai!

## Actor-Critic Model

Actor-Critic model policy gradients ka ek aur evolution hai, jisme hum ek neural network banate hain jo policy aur estimated rewards, dono ko seekhta hai. Is network ke do outputs honge (ya aap ise do alag networks ke roop mein dekh sakte hain):
* **Actor** hamein state probability distribution dekar action recommend karega, jaisa policy gradient model mein hota hai.
* **Critic** estimate karega ki un actions se reward kya hoga. Yeh diye gaye state par future mein total estimated rewards return karta hai.

Chaliye aisa ek model define karte hain:

In [ ]:
from itertools import count
import torch.nn.functional as F

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
env = gym.make("CartPole-v1")

state_size = env.observation_space.shape[0]
action_size = env.action_space.n
lr = 0.0001

class Actor(torch.nn.Module):
    def __init__(self, state_size, action_size):
        super(Actor, self).__init__()
        self.state_size = state_size
        self.action_size = action_size
        self.linear1 = torch.nn.Linear(self.state_size, 128)
        self.linear2 = torch.nn.Linear(128, 256)
        self.linear3 = torch.nn.Linear(256, self.action_size)

    def forward(self, state):
        output = F.relu(self.linear1(state))
        output = F.relu(self.linear2(output))
        output = self.linear3(output)
        distribution = torch.distributions.Categorical(F.softmax(output, dim=-1))
        return distribution


class Critic(torch.nn.Module):
    def __init__(self, state_size, action_size):
        super(Critic, self).__init__()
        self.state_size = state_size
        self.action_size = action_size
        self.linear1 = torch.nn.Linear(self.state_size, 128)
        self.linear2 = torch.nn.Linear(128, 256)
        self.linear3 = torch.nn.Linear(256, 1)

    def forward(self, state):
        output = F.relu(self.linear1(state))
        output = F.relu(self.linear2(output))
        value = self.linear3(output)
        return value

Hamein apne `discounted_rewards` aur `run_episode` functions ko thoda modify karna padega:

In [ ]:
def discounted_rewards(next_value, rewards, masks, gamma=0.99):
    R = next_value
    returns = []
    for step in reversed(range(len(rewards))):
        R = rewards[step] + gamma * R * masks[step]
        returns.insert(0, R)
    return returns

def run_episode(actor, critic, n_iters):
    optimizerA = torch.optim.Adam(actor.parameters())
    optimizerC = torch.optim.Adam(critic.parameters())
    for iter in range(n_iters):
        state = env.reset()
        log_probs = []
        values = []
        rewards = []
        masks = []
        entropy = 0
        env.reset()

        for i in count():
            env.render()
            state = torch.FloatTensor(state).to(device)
            dist, value = actor(state), critic(state)

            action = dist.sample()
            next_state, reward, done, _ = env.step(action.cpu().numpy())

            log_prob = dist.log_prob(action).unsqueeze(0)
            entropy += dist.entropy().mean()

            log_probs.append(log_prob)
            values.append(value)
            rewards.append(torch.tensor([reward], dtype=torch.float, device=device))
            masks.append(torch.tensor([1-done], dtype=torch.float, device=device))

            state = next_state

            if done:
                print('Iteration: {}, Score: {}'.format(iter, i))
                break


        next_state = torch.FloatTensor(next_state).to(device)
        next_value = critic(next_state)
        returns = discounted_rewards(next_value, rewards, masks)

        log_probs = torch.cat(log_probs)
        returns = torch.cat(returns).detach()
        values = torch.cat(values)

        advantage = returns - values

        actor_loss = -(log_probs * advantage.detach()).mean()
        critic_loss = advantage.pow(2).mean()

        optimizerA.zero_grad()
        optimizerC.zero_grad()
        actor_loss.backward()
        critic_loss.backward()
        optimizerA.step()
        optimizerC.step()


Ab hum main training loop chalayenge. Hum manual network training process use karenge, jisme hum appropriate loss function calculate karenge aur network parameters ko update karenge:

In [ ]:

actor = Actor(state_size, action_size).to(device)
critic = Critic(state_size, action_size).to(device)
run_episode(actor, critic, n_iters=100)

In [ ]:
env.close()

## Key Takeaways

Humne is demo mein do RL algorithms dekhe: simple policy gradient aur zyada sophisticated actor-critic. Aap dekh sakte hain ki yeh algorithms state, action aur reward jaise abstract ideas ke saath kaam karte hain - isliye inhe bahut alag-alag environments mein apply kiya ja sakta hai.

Reinforcement Learning hamein sirf final reward ko dekhkar problem solve karne ki best strategy seekhne ki permission deta hai. Yeh fact ki hamein labeled dataset ki zaroorat nahi hoti, hamein apne models ko optimize karne ke liye simulations ko kai baar repeat karne ki suvidha deta hai. Haalaanki, RL mein abhi bhi kaafi challenges hain, jinhe aap seekh sakte hain agar aap AI ke is interesting field par zyada focus karne ka decision lein.

---

<!-- HINGLISH NOTE START -->
**Note**:
Yeh document Hindi translation ko Hinglish (Roman script) mein convert karke banaya gaya hai, taaki padhna aur samajhna aasaan ho.
Isme kuch galtiyan ya anuvaad ki kamiyan ho sakti hain. Sabse sahi jaankari ke liye original English document ko hi authoritative source maanein.
<!-- HINGLISH NOTE END -->